In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np

from Data_query.trino_config import *

In [ ]:
stop_trino()

In [3]:
big_workers = 0
workers = 1
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count=workers, big_worker_desired_count=big_workers)
sleep(30)

Trino service is already running.


In [2]:
# counting the number of conformant and nonconformant sites for the same OEM and install date quarter excluding generic inverters
df = iceberg_sql(""" 
            with data as (
                select  
                        c.site_id, 
                        arbitrary(m.manufacturer) as OEM,
                        arbitrary(m.install_date) as install_date,
                        sum(nonconformance_voltvar_red_sum) as nonconf_sum, 
                        sum(nonconformance_voltvar_red_count) as nonconf_count,
                        sum(nonconformance_voltvar_red_count)/sum(cast (total_count as double)) as nonconf_ratio 
                from conformance_voltvar as c
                    join (select distinct site_id, manufacturer, 
                    concat(cast(year(pv_install_date) as varchar), '-Q', cast(quarter(pv_install_date) as varchar)) as install_date
                        from meta_up23c where pv_install_date >= date '2024-01-01') as m
                    on c.site_id = m.site_id
                group by c.site_id
             ),
            conf_data as (
                select * from data where nonconf_ratio <= .1
            ),
            nonconf_data as (
                select * from data where nonconf_ratio > .1
            )
            select coalesce(nd.OEM, cd.OEM) as OEM, nd.install_date,  
                 count(distinct cd.site_id) as num_conf_sites,
                 count(distinct nd.site_id) as num_nonconf_sites,
                cast (count(DISTINCT nd.site_id) as double)/(cast(count(DISTINCT cd.site_id) + count(DISTINCT nd.site_id) as double))*100 AS "nonconf_install_ratio%"
            from nonconf_data nd FULL outer join conf_data cd on nd.OEM = cd.OEM and nd.install_date = cd.install_date
            where nd.OEM != 'Generic Inverter'
            group by coalesce(nd.OEM, cd.OEM), nd.install_date
            order by coalesce(nd.OEM, cd.OEM), nd.install_date
            """)
# df.groupby(['OEM', 'install_date']).agg({'conf_site_id': lambda x: x.explode().count(), 'nonconf_site_id': lambda x: x.explode().count()}, axis=0)
df.round(1)

,OEM,install_date,num_conf_sites,num_nonconf_sites,nonconf_install_ratio%
0,ABB,2024-Q1,2,5,71.4
1,ABB,2024-Q3,1,1,50.0
2,Fimer,2024-Q3,0,1,100.0
3,Fronius,2024-Q1,19,4,17.4
4,Fronius,2024-Q2,19,1,5.0
5,Fronius,2024-Q3,12,8,40.0
6,Fronius,2024-Q4,16,2,11.1
7,Fronius,2025-Q1,18,2,10.0
8,Fronius,2025-Q2,9,1,10.0
9,Fronius,2025-Q3,2,1,33.3
